# Seasonal storage: keeping the energy in calendar order

Suppose a region wants to cover its whole annual load with solar PV, and to store the summer
surplus for the winter. How large must that seasonal storage be? A model that uses typical days
finds out by stepping through the calendar day by day, with each day replaced by its typical
day. Whether the storage comes out right depends on how the energy adds up over the seasons, not
on how well each hour fits.

In this tutorial we size such a storage from a year of hourly data, then from 8 typical days
found by three clustering methods. By the end you will have seen that

* `kmeans`, which fits each hour best, makes the storage too small,
* `averaging`, which fits each hour worse, sizes it almost exactly, and
* `contiguous` only gets it right with `representation="mean"`.

This is part 2 of two [clustering tutorials](../index.md#clustering). Each part stands on its own.

## 1  A year of solar and load

We take the whole year of hourly solar irradiance (`GHI`) and electricity `Load` from the
example data. The PV is sized so that over the year it produces exactly as much as the load.
Every hour, the storage takes the surplus, PV minus load, or covers the deficit. Its level is the
running sum of that balance, and it must hold the whole swing between its lowest and its highest
level. We keep this storage model simple on purpose: no losses, no other power plants.

Every run reduces the year to `N_TYPICAL` typical days with the same settings, so we write them
down once in a small function; only the clustering will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as each representation
produced them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig
from tsam.plot import compare_series

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw[["GHI", "Load"]]
PV_FACTOR = data["Load"].sum() / data["GHI"].sum()  # MW of PV output per W/m²
N_TYPICAL = 8
UNITS = {"Storage level": "MWh"}
COLORS = {
    "original": "#000000",
    "kmeans": "#0072B2",
    "averaging": "#009E73",
    "contiguous": "#AD3977",
    "contiguous + mean": "#8759A5",
}


def storage_level(frame):
    balance = PV_FACTOR * frame["GHI"] - frame["Load"]
    return balance.cumsum().to_frame("Storage level")


def storage_needed(level):
    return level["Storage level"].max() - level["Storage level"].min()


def aggregate(cluster):
    np.random.seed(0)  # kmeans starts from random centers
    return tsam.aggregate(
        data,
        n_clusters=N_TYPICAL,
        period_duration="1D",
        cluster=cluster,
        preserve_column_means=False,
    )


def sizes(levels):
    needed = pd.Series({name: storage_needed(level) for name, level in levels.items()})
    return pd.DataFrame(
        {
            "storage needed (MWh)": needed.round(-3).astype(int),
            "error (%)": (100 * (needed / needed["original"] - 1)).round(1),
        }
    )

Size the storage from the original year:

In [ ]:
original = storage_level(data)
levels = {"original": original}

print(f"PV: {PV_FACTOR * 1000:,.0f} MW at 1000 W/m²")
compare_series(
    levels, units=UNITS, colors=COLORS, title="Storage level over the year"
).show()
sizes(levels)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
level = original["Storage level"]
assert str(level.idxmin().date()) == "2010-03-21"
assert str(level.idxmax().date()) == "2010-09-26"
assert round(storage_needed(original) / 1e6, 2) == 1.21

The level falls through the winter to its lowest point on 21 March, rises through the summer to
its highest point on 26 September, and falls back to where it started. The storage has to hold
the whole swing: about 1.21 million MWh.

## 2  Typical days by similarity: `kmeans`

`kmeans` finds the typical days that come closest to the days they stand for
([part 1](01_similar_days.ipynb) compares it with the other similarity methods).
`result.reconstructed` lays the typical days out in calendar order, one for every day they stand
for: exactly the sequence a model with seasonal storage steps through.

In [ ]:
kmeans = aggregate(ClusterConfig(method="kmeans"))
levels["kmeans"] = storage_level(kmeans.reconstructed)

compare_series(
    levels, units=UNITS, colors=COLORS, title="Storage level over the year"
).show()
sizes(levels)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert len(kmeans.reconstructed) == len(data)
assert sizes(levels).loc["kmeans", "error (%)"].round() == -7

The `kmeans` level follows the seasons, but its swing is smaller: the storage comes out 7% too
small. `kmeans` groups similar days from different seasons, a sunny day in April with one in
August, so each typical day blurs a little of the seasonal swing.

## 3  Typical days by calendar: `averaging`

`averaging` ignores similarity. It cuts the year into blocks of consecutive days and represents
each block by its mean day:

In [ ]:
averaging = aggregate(ClusterConfig(method="averaging"))
levels["averaging"] = storage_level(averaging.reconstructed)

compare_series(
    levels, units=UNITS, colors=COLORS, title="Storage level over the year"
).show()
sizes(levels)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert sizes(levels).loc["averaging", "error (%)"].round() == -2

The `averaging` level stays close to the original, and the storage comes out within 2%. That is
surprising at first, because its typical days fit each hour much worse than those of `kmeans`.
Compare how far each comes from the original hours, in normalized units (lower is better):

In [ ]:
pd.Series(
    {
        "kmeans": kmeans.accuracy.weighted_rmse,
        "averaging": averaging.accuracy.weighted_rmse,
    },
    name="hourly RMSE",
).round(3)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert averaging.accuracy.weighted_rmse > 1.5 * kmeans.accuracy.weighted_rmse

The reason is in the blocks. A block's mean day holds exactly the block's energy, so at the end of
every block the storage level is exactly right again. Check the difference to the original at the
last hour of each block:

In [ ]:
last_days = np.flatnonzero(np.diff(averaging.cluster_assignments, append=-1) != 0)
last_hours = (last_days + 1) * 24 - 1
(levels["averaging"] - original).iloc[last_hours].round().astype(int)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
gap = (levels["averaging"] - original)["Storage level"].iloc[last_hours]
assert (gap.abs() < 1).all() and len(gap) == N_TYPICAL

Within a block, the level can drift away a little. But the errors cannot pile up over the
seasons.

## 4  Calendar blocks by similarity: `contiguous`

`contiguous` also builds blocks of consecutive days, but it chooses them by similarity: it only
merges neighbouring days that are alike. Try it with its default representation:

In [ ]:
contiguous = aggregate(ClusterConfig(method="contiguous"))
levels["contiguous"] = storage_level(contiguous.reconstructed)

compare_series(
    levels, units=UNITS, colors=COLORS, title="Storage level over the year"
).show()
sizes(levels)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert sizes(levels).loc["contiguous", "error (%)"].round() == -28

The storage comes out 28% too small. By default, `contiguous` represents each block by its
medoid, one real day of the block, and a single day doesn't hold the block's energy. Use the
block's mean instead:

In [ ]:
contiguous_mean = aggregate(ClusterConfig(method="contiguous", representation="mean"))
levels["contiguous + mean"] = storage_level(contiguous_mean.reconstructed)

compare_series(
    levels, units=UNITS, colors=COLORS, title="Storage level over the year"
).show()
sizes(levels)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
table = sizes(levels)
assert table.loc["contiguous + mean", "error (%)"].round() == -5
errors = table["error (%)"].abs()
assert (
    errors["averaging"]
    < errors["contiguous + mean"]
    < errors["kmeans"]
    < errors["contiguous"]
)
rescaled = tsam.aggregate(
    data,
    n_clusters=N_TYPICAL,
    period_duration="1D",
    cluster=ClusterConfig(method="contiguous"),
)
error = (
    storage_needed(storage_level(rescaled.reconstructed)) / storage_needed(original) - 1
)
assert round(100 * error) == -7

Now the storage comes out 5% too small: better than `kmeans`, though not as close as
`averaging`. With tsam's default rescaling switched on, the medoid version comes out 7% too
small: better than without, but still not as close as `averaging`.

## 5  Try it yourself

Change `N_TYPICAL` in the setup cell, for example to 4 or 24, and run the notebook again. With
more typical days, all methods come closer to the real storage; in our runs, `averaging` stayed
the closest up to 48 typical days.

## What you've learned

* A seasonal storage is sized by how the energy adds up over the calendar, not by how well each
  hour fits.
* `kmeans` fits each hour best, but it mixes days from different seasons, and the storage comes
  out too small.
* `averaging` fits each hour worse, but each block's mean day keeps the block's energy: the
  storage level is exactly right at every block end, and the storage size within 2%.
* `contiguous` also builds blocks, but only with `representation="mean"` does each typical day
  keep its block's energy.

Where to go next:

* [Clustering methods](../../how-to/clustering_methods.ipynb) — pick a method by what your model
  needs.
* [Working with typical periods](../../how-to/working_with_typical_periods.ipynb) — use the
  assignments for storage across periods.
* [Part 1: As similar as possible](01_similar_days.ipynb) — what the extra runtime of `kmeans`
  and `kmedoids` buys.